# AURA Voice — F5-TTS Irish Voice Clone (Colab)

This notebook runs **F5-TTS** zero-shot voice cloning on a free T4 GPU.
It exposes a public API endpoint via Gradio that your AURA app can connect to.

**Steps:**
1. Run all cells (5 minutes total)
2. Copy the public Gradio URL from Cell 5
3. Paste it into your .env as AURA_XTTS_URL

In [ ]:
# Cell 1 — Install F5-TTS + dependencies (~2 min)
!pip install -q f5-tts gradio scipy numpy

In [ ]:
import os
from google.colab import files

os.makedirs('voices', exist_ok=True)

if not os.path.exists('voices/aura_irish.wav'):
    print('Downloading Irish voice reference from GitHub...')
    !wget -q -O voices/aura_irish.wav https://raw.githubusercontent.com/shelfgenius/permanentai-os/main/voices/aura_irish.wav
    if os.path.exists('voices/aura_irish.wav') and os.path.getsize('voices/aura_irish.wav') > 1000:
        print(f'Downloaded! Size: {os.path.getsize("voices/aura_irish.wav"):,} bytes')
    else:
        print('Download failed. Upload manually:')
        uploaded = files.upload()
        for name, data in uploaded.items():
            with open('voices/aura_irish.wav', 'wb') as f:
                f.write(data)
            print(f'Saved {name} as voices/aura_irish.wav')
else:
    print(f'Voice reference already exists ({os.path.getsize("voices/aura_irish.wav"):,} bytes)')

In [ ]:
import torch
import soundfile as sf
from f5_tts.api import F5TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('Loading F5-TTS model (first run downloads ~1.2GB)...')
tts = F5TTS(model='F5TTS_v1_Base', device=device)
print('F5-TTS model loaded!')

In [ ]:
import IPython.display as ipd

wav, sr, _ = tts.infer(
    ref_file='voices/aura_irish.wav',
    ref_text='',
    gen_text='Hello! I am Aura, your AI assistant. How can I help you today?',
)

sf.write('test_output.wav', wav, sr)
print(f'Generated audio: {len(wav)/sr:.1f}s at {sr}Hz')
ipd.display(ipd.Audio('test_output.wav', autoplay=True))

In [ ]:
import gradio as gr
import tempfile
import soundfile as sf

def synthesize(text):
    """Generate speech cloning the Irish voice."""
    if not text or not text.strip():
        return None
    wav, sr, _ = tts.infer(
        ref_file='voices/aura_irish.wav',
        ref_text='',
        gen_text=text.strip(),
    )
    tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    sf.write(tmp.name, wav, sr)
    return tmp.name

def health():
    return {'status': 'ready', 'device': device, 'model': 'f5-tts'}

with gr.Blocks(title='AURA Voice') as demo:
    gr.Markdown('# AURA Voice — Irish Voice Clone (F5-TTS on Colab T4)')
    text_in = gr.Textbox(label='Text', value='Hello, I am Aura.', lines=3)
    btn = gr.Button('Speak', variant='primary')
    audio_out = gr.Audio(label='Output', type='filepath')
    btn.click(fn=synthesize, inputs=[text_in], outputs=audio_out, api_name='synthesize')
    health_btn = gr.Button('Health Check', size='sm')
    health_json = gr.JSON(label='Status')
    health_btn.click(fn=health, outputs=health_json, api_name='health')

print('=' * 60)
print('  AURA Voice server starting...')
print('  Copy the PUBLIC URL below and set as AURA_XTTS_URL in .env')
print('=' * 60)

demo.launch(share=True, quiet=False)